# 05（US） 一键生成结论报告（从 `data/clean_us/` 汇总）

目标：把 **口径 → 主结论（03）→ 稳健性（04）→ 风险提示与下一步** 汇总成一个可读的报告。

本 notebook **不联网**、不调用 Tushare，只读取前序 notebooks 的落盘文件。

运行顺序（VS Code Run All）：
- `01_fetch_market_data` → `02_build_ganzhi_calendar` → `03_ganzhi_effect_analysis` →（可选）`04a_controls_models`…`04f_resonance_harmonics`（索引：`04_robustness_and_modeling`）→ `05_report`

输出：
- `data/clean_us/report/report_*.md`（Markdown 报告）
- `data/clean_us/report/figures/*.png`（报告图表）


## 重要说明（解读边界）

- 这是一个 **多重比较**（10 天干 / 12 地支 / 60 甲子 × 多指数 × 多指标）的研究问题；任何“显著”必须同时报告 **效应大小** 与 **FDR(BH) q 值**。
- 当前结论优先回答“是否存在统计关系、是否稳健”，**不等价于可交易策略**。
- 本项目把日柱日界固定为 **北京时间自然日（00:00~24:00）**，与交易日日期对齐。
- 研究分层：只有通过确认性门槛的才写成“正式结论”；Phase 2（`day_group × year_element`）采用 **全局 gate → 才看局部 cell** 的层级检验（见 `docs/Protocol.md`）。


In [ ]:
from __future__ import annotations

import textwrap
import warnings
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats


warnings.filterwarnings('ignore')



from matplotlib import font_manager, rcParams


def set_chinese_font() -> None:
    # Best-effort Chinese font setup for Matplotlib.
    # - Tries common Linux CJK fonts
    # - On WSL, also tries registering Windows fonts under /mnt/c/Windows/Fonts

    candidates = [
        'Noto Sans CJK SC',
        'Noto Sans CJK TC',
        'Noto Sans CJK JP',
        'Source Han Sans SC',
        'WenQuanYi Micro Hei',
        'Microsoft YaHei',
        'SimHei',
        'SimSun',
        'Arial Unicode MS',
    ]

    # WSL fallback: register Windows fonts if available
    try:
        from pathlib import Path as _Path

        win_dir = _Path('/mnt/c/Windows/Fonts')
        win_files = [
            win_dir / 'msyh.ttc',
            win_dir / 'msyhbd.ttc',
            win_dir / 'simhei.ttf',
            win_dir / 'simsun.ttc',
            win_dir / 'arialuni.ttf',
        ]

        extra_names: list[str] = []
        addfont = getattr(font_manager.fontManager, 'addfont', None)
        for fp in win_files:
            try:
                if fp.is_file():
                    if callable(addfont):
                        addfont(str(fp))
                    extra_names.append(font_manager.FontProperties(fname=str(fp)).get_name())
            except Exception:
                continue

        candidates = [n for n in extra_names if n] + candidates
    except Exception:
        pass

    for name in candidates:
        try:
            font_manager.findfont(name, fallback_to_default=False)
            rcParams['font.sans-serif'] = [name]
            rcParams['axes.unicode_minus'] = False
            print(f'Using Chinese font: {name}')
            return
        except Exception:
            continue

    rcParams['axes.unicode_minus'] = False
    print(
        'Warning: no Chinese font found; Chinese labels may not render. '
        'If you are on WSL, ensure Windows fonts are accessible under /mnt/c/Windows/Fonts; '
        'or install Noto CJK fonts in Linux (e.g., fonts-noto-cjk).'
    )


set_chinese_font()

def find_project_root(start: Path | None = None) -> Path:
    here = (start or Path.cwd()).resolve()

    for candidate in [here] + list(here.parents):
        if (candidate / 'AGENTS.md').is_file() and (candidate / 'data').is_dir() and (candidate / 'notebooks').is_dir():
            return candidate

    for candidate in here.glob('*'):
        if candidate.is_dir() and (candidate / 'AGENTS.md').is_file() and (candidate / 'data').is_dir() and (candidate / 'notebooks').is_dir():
            return candidate

    return here


def set_cn_font_fallback() -> None:
    # 尽量使用常见中文字体；若机器缺字库，仍可能出现方块字。
    plt.rcParams['font.sans-serif'] = [
        'Microsoft YaHei',
        'SimHei',
        'PingFang SC',
        'Noto Sans CJK SC',
        'WenQuanYi Micro Hei',
        'Arial Unicode MS',
        'DejaVu Sans',
    ]
    plt.rcParams['axes.unicode_minus'] = False


ROOT = find_project_root()
print('PROJECT_ROOT =', ROOT)

CLEAN_DIR = ROOT / 'data/clean_us'
ROBUST_DIR = CLEAN_DIR / 'robustness'
REPORT_DIR = CLEAN_DIR / 'report'
FIG_DIR = REPORT_DIR / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

RUN_TIMESTAMP = datetime.now().strftime('%Y%m%d_%H%M%S')
REPORT_PATH = REPORT_DIR / f'report_{RUN_TIMESTAMP}.md'

ALPHA_Q = 0.10

TS_CODES_DEFAULT = ['spx', 'ndq', 'ndx', 'dji']
STEMS_ORDER = list('甲乙丙丁戊己庚辛壬癸')
BRANCHES_ORDER = list('子丑寅卯辰巳午未申酉戌亥')
YEAR_ELEMENT_ORDER = ['木', '火', '土', '金', '水']
PHASE2_DAY_GROUPS = ['stem', 'branch', 'ganzhi_day']

sns.set_theme(style='whitegrid', font_scale=1.0)
set_cn_font_fallback()


def detect_ts_codes() -> list[str]:
    ts_codes = []
    for ts_code in TS_CODES_DEFAULT:
        if (CLEAN_DIR / f'market_ganzhi_{ts_code}.csv.gz').exists():
            ts_codes.append(ts_code)
    return ts_codes


TS_CODES = ['spx', 'ndq', 'ndx', 'dji']
if not TS_CODES:
    raise FileNotFoundError(
        '未找到 market_ganzhi_*.csv.gz。请先 Run All notebooks 01-03（至少产出 data/clean_us/market_ganzhi_{ts_code}.csv.gz）。'
    )

print('TS_CODES =', TS_CODES)
print('REPORT_PATH =', REPORT_PATH)


In [ ]:
def safe_read_csv(path: Path, **kwargs) -> pd.DataFrame:
    if not path.exists():
        return pd.DataFrame()
    if path.suffix == '.gz':
        kwargs.setdefault('compression', 'gzip')
    return pd.read_csv(path, **kwargs)


def parse_trade_date(s: str) -> pd.Timestamp:
    return pd.to_datetime(str(s), format='%Y%m%d')


def format_pct(x: float, digits: int = 2) -> str:
    if x is None or (isinstance(x, float) and not np.isfinite(x)):
        return ''
    return f'{x * 100:.{digits}f}%'


def format_bp(x: float, digits: int = 1) -> str:
    # basis points for daily returns (1bp = 0.01%)
    if x is None or (isinstance(x, float) and not np.isfinite(x)):
        return ''
    return f'{x * 10000:.{digits}f}bp'


def df_to_markdown_table(df: pd.DataFrame, max_rows: int = 50) -> str:
    if df is None or df.empty:
        return '(空表)'

    view = df.copy()
    if len(view) > max_rows:
        view = view.head(max_rows)

    cols = [str(c) for c in view.columns]
    lines = []
    lines.append('| ' + ' | '.join(cols) + ' |')
    lines.append('| ' + ' | '.join(['---'] * len(cols)) + ' |')
    for _, row in view.iterrows():
        vals = []
        for c in view.columns:
            v = row[c]
            if isinstance(v, float):
                if np.isfinite(v):
                    vals.append(f'{v:.6g}')
                else:
                    vals.append('')
            else:
                vals.append(str(v) if v is not None else '')
        lines.append('| ' + ' | '.join(vals) + ' |')

    if len(df) > max_rows:
        lines.append(f'\n> 注：仅展示前 {max_rows} 行（共 {len(df)} 行）。')
    return '\n'.join(lines)




def fdr_bh(pvals: np.ndarray) -> np.ndarray:
    # Benjamini-Hochberg FDR q-values for a 1D array.
    p = np.asarray(pvals, dtype=float)
    n = p.size
    order = np.argsort(p)
    q = np.empty(n, dtype=float)
    prev = 1.0

    for rank in range(n - 1, -1, -1):
        i = order[rank]
        val = p[i] * n / (rank + 1)
        prev = min(prev, val)
        q[i] = prev

    return np.clip(q, 0.0, 1.0)

def save_figure(fig: plt.Figure, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=160, bbox_inches='tight')


def read_main_tests(ts_code: str) -> pd.DataFrame:
    path = CLEAN_DIR / f'ganzhi_tests_{ts_code}.csv'
    return safe_read_csv(path)


def read_main_stats(ts_code: str) -> pd.DataFrame:
    path = CLEAN_DIR / f'ganzhi_stats_{ts_code}.csv'
    return safe_read_csv(path)


def read_ganzhi_calendar() -> pd.DataFrame:
    path = CLEAN_DIR / 'ganzhi_trade_dates.csv.gz'
    return safe_read_csv(path, dtype={'trade_date': str})


def read_controls(ts_code: str, group_col: str, target: str) -> pd.DataFrame:
    path = ROBUST_DIR / f'controls_{ts_code}_{group_col}_{target}.csv'
    return safe_read_csv(path)


def read_perm_global() -> pd.DataFrame:
    return safe_read_csv(ROBUST_DIR / 'perm_global.csv')


def read_subsample_bing() -> pd.DataFrame:
    return safe_read_csv(ROBUST_DIR / 'subsample_bing.csv')


def read_walk_forward(ts_code: str) -> pd.DataFrame:
    path = ROBUST_DIR / f'walk_forward_{ts_code}_stem.csv'
    return safe_read_csv(path)


def read_perm_global_controls_resid() -> pd.DataFrame:
    return safe_read_csv(ROBUST_DIR / 'perm_global_controls_resid.csv')


def read_hac_sensitivity(ts_code: str) -> pd.DataFrame:
    path = ROBUST_DIR / f'hac_sensitivity_controls_{ts_code}_stem_ret_1d.csv'
    return safe_read_csv(path)


def read_block_bootstrap(ts_code: str) -> pd.DataFrame:
    path = ROBUST_DIR / f'block_bootstrap_controls_resid_{ts_code}_stem_ret_1d.csv'
    return safe_read_csv(path)


def read_walk_forward_oos_effects(ts_code: str) -> pd.DataFrame:
    path = ROBUST_DIR / f'walk_forward_{ts_code}_stem_oos_effects.csv'
    return safe_read_csv(path)


def read_walk_forward_oos_effects_controls(ts_code: str) -> pd.DataFrame:
    path = ROBUST_DIR / f'walk_forward_{ts_code}_stem_oos_effects_controls.csv'
    return safe_read_csv(path)


def read_walk_forward_oos_effects_controls_sweep(ts_code: str) -> pd.DataFrame:
    path = ROBUST_DIR / f'walk_forward_{ts_code}_stem_oos_effects_controls_sweep.csv'
    return safe_read_csv(path)


def read_phase2_gate_summary() -> pd.DataFrame:
    return safe_read_csv(ROBUST_DIR / 'interaction_gate_summary_ret_1d.csv')


def read_phase2_meta(day_group: str) -> pd.DataFrame:
    path = ROBUST_DIR / f'meta_interaction_{day_group}_ret_1d.csv'
    return safe_read_csv(path)


def read_phase2_cell_effects(ts_code: str, day_group: str) -> pd.DataFrame:
    path = ROBUST_DIR / f'interaction_cell_effects_{ts_code}_{day_group}_ret_1d.csv'
    return safe_read_csv(path)



## 1) 数据覆盖（用于确认样本范围与日收益口径）


In [ ]:
coverage_records = []
market_end_dates = []

for ts_code in TS_CODES:
    df = safe_read_csv(CLEAN_DIR / f'market_ganzhi_{ts_code}.csv.gz', dtype={'trade_date': str})
    if df.empty:
        continue
    df = df.dropna(subset=['ret_1d', 'up'])
    start_date = parse_trade_date(df['trade_date'].min()).date()
    end_date = parse_trade_date(df['trade_date'].max()).date()
    market_end_dates.append(end_date)

    coverage_records.append(
        {
            'ts_code': ts_code,
            'n_days': int(len(df)),
            'start_date': str(start_date),
            'end_date': str(end_date),
            'mean_ret_1d': float(df['ret_1d'].mean()),
            'std_ret_1d': float(df['ret_1d'].std(ddof=1)),
            'p_up': float(df['up'].mean()),
        }
    )

coverage_df = pd.DataFrame.from_records(coverage_records).sort_values('ts_code')
if coverage_df.empty:
    raise ValueError('market_ganzhi 文件为空或缺失 ret_1d/up。请先完整 Run All notebooks 01-03。')
from IPython.display import display

display(coverage_df)

# 干支日历（交易日）sanity check：分布是否大致均匀、起止日期是否匹配
ganzhi_cal = read_ganzhi_calendar()
if ganzhi_cal.empty:
    print('缺少 ganzhi_trade_dates.csv.gz（如未运行 02，可忽略；但建议补齐以确保日柱口径一致）。')
else:
    ganzhi_cal = ganzhi_cal.dropna(subset=['trade_date', 'stem', 'branch']).copy()
    print('ganzhi_trade_dates range =', ganzhi_cal['trade_date'].min(), '->', ganzhi_cal['trade_date'].max())

    stem_counts = ganzhi_cal['stem'].value_counts().reindex(STEMS_ORDER)
    branch_counts = ganzhi_cal['branch'].value_counts().reindex(BRANCHES_ORDER)

    stem_dist = stem_counts.reset_index()
    stem_dist.columns = ['stem', 'n_days']
    stem_dist['share'] = stem_dist['n_days'] / stem_dist['n_days'].sum()

    branch_dist = branch_counts.reset_index()
    branch_dist.columns = ['branch', 'n_days']
    branch_dist['share'] = branch_dist['n_days'] / branch_dist['n_days'].sum()

    display(stem_dist)
    display(branch_dist)


## 2) 主结论（Notebook 03：无控制变量、样本内）

说明：该部分来自 `ganzhi_tests_{ts_code}.csv`（已做 BH-FDR）。这里主要聚焦：
- `stem × mean_ret`（10 个天干）
- 是否存在 `q <= 0.1` 的信号


In [ ]:
main_bing_rows = []
main_sig_rows = []

for ts_code in TS_CODES:
    tests = read_main_tests(ts_code)
    if tests.empty:
        continue

    stem_ret = tests[(tests['group_type'] == 'stem') & (tests['metric'] == 'mean_ret')].copy()
    if stem_ret.empty:
        continue

    # 仅保留排序用字段
    stem_ret = stem_ret[['group_value', 'effect', 'p_value', 'q_value', 'n']].copy()

    # 丙的记录
    bing = stem_ret[stem_ret['group_value'] == '丙']
    if len(bing) == 1:
        r = bing.iloc[0]
        main_bing_rows.append(
            {
                'ts_code': ts_code,
                'effect_bing_minus_all': float(r['effect']),
                'q_value': float(r['q_value']),
                'p_value': float(r['p_value']),
                'n_bing': int(r['n']),
            }
        )

    # 显著项（按 q<=阈值）
    sig = stem_ret[stem_ret['q_value'] <= ALPHA_Q].sort_values(['q_value', 'p_value'])
    for _, r in sig.iterrows():
        main_sig_rows.append(
            {
                'ts_code': ts_code,
                'group_value': r['group_value'],
                'effect': float(r['effect']),
                'p_value': float(r['p_value']),
                'q_value': float(r['q_value']),
                'n': int(r['n']),
            }
        )

main_bing_df = pd.DataFrame.from_records(main_bing_rows)
if main_bing_df.empty:
    main_bing_df = pd.DataFrame(columns=['ts_code', 'effect_bing_minus_all', 'q_value', 'p_value', 'n_bing'])
else:
    main_bing_df = main_bing_df.sort_values('ts_code')

main_sig_df = pd.DataFrame.from_records(main_sig_rows)
if main_sig_df.empty:
    main_sig_df = pd.DataFrame(columns=['ts_code', 'group_value', 'effect', 'p_value', 'q_value', 'n'])
else:
    main_sig_df = main_sig_df.sort_values(['ts_code', 'q_value', 'p_value'])

from IPython.display import display

print('主分析（03，无控制）: stem×mean_ret 中 q<=', ALPHA_Q, '的项（若为空则表示未通过阈值）')
if main_sig_df.empty:
    print('(无)')
else:
    display(main_sig_df)

# 全量扫描：每个 group_type×metric 的最小 q（用于快速复盘；不代表通过阈值）
top_scan_rows = []
required = {'group_type', 'metric', 'group_value', 'effect', 'p_value', 'q_value', 'n'}

for ts_code in TS_CODES:
    tests = read_main_tests(ts_code)
    if tests.empty or (not required.issubset(set(tests.columns))):
        continue

    for (gt, metric), sub in tests.groupby(['group_type', 'metric']):
        sub = sub.sort_values(['q_value', 'p_value'])
        if sub.empty:
            continue
        r = sub.iloc[0]
        top_scan_rows.append(
            {
                'ts_code': ts_code,
                'group_type': gt,
                'metric': metric,
                'group_value': r['group_value'],
                'effect': float(r['effect']),
                'p_value': float(r['p_value']),
                'q_value': float(r['q_value']),
                'n': int(r['n']),
            }
        )

top_scan_df = pd.DataFrame.from_records(top_scan_rows)
if top_scan_df.empty:
    top_scan_df = pd.DataFrame(columns=['ts_code', 'group_type', 'metric', 'group_value', 'effect', 'p_value', 'q_value', 'n'])
else:
    top_scan_df = top_scan_df.sort_values(['ts_code', 'metric', 'group_type', 'q_value', 'p_value'])

top_scan_pass_df = top_scan_df[top_scan_df['q_value'] <= ALPHA_Q].copy()

print('全量扫描：每个 group_type×metric 里 q 最小的项（不代表通过阈值）')
display(top_scan_df)

print('通过阈值的项（q<=阈值）')
if top_scan_pass_df.empty:
    print('(无)')
else:
    display(top_scan_pass_df)


In [ ]:
# 图：03（无控制）stem 的 mean_ret effect（组均值 - 全样本均值）

n_plots = len(TS_CODES)
ncols = 2
nrows = int(np.ceil(n_plots / ncols))
fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(14, 4.5 * nrows), constrained_layout=True)
axes = np.atleast_1d(axes).ravel()

for i, ts_code in enumerate(TS_CODES):
    ax = axes[i]
    tests = read_main_tests(ts_code)
    stem_ret = tests[(tests['group_type'] == 'stem') & (tests['metric'] == 'mean_ret')].copy()
    if stem_ret.empty:
        ax.set_title(f'{ts_code}（缺少 tests）')
        ax.axis('off')
        continue

    stem_ret['group_value'] = pd.Categorical(stem_ret['group_value'], categories=STEMS_ORDER, ordered=True)
    stem_ret = stem_ret.sort_values('group_value')

    colors = ['tab:red' if str(v) == '丙' else 'tab:blue' for v in stem_ret['group_value'].astype(str).tolist()]
    ax.bar(stem_ret['group_value'].astype(str), stem_ret['effect'], color=colors, alpha=0.85)
    ax.axhline(0.0, color='black', lw=1)
    ax.set_title(f'03 无控制：stem 对 mean_ret 的 effect（{ts_code}）')
    ax.set_ylabel('effect (ret_1d)')
    ax.set_xlabel('stem')

    # 标注通过阈值的 q
    for x, eff, qv in zip(stem_ret['group_value'].astype(str), stem_ret['effect'], stem_ret['q_value']):
        if np.isfinite(qv) and qv <= ALPHA_Q:
            ax.text(x, eff, f'q={qv:.3g}', ha='center', va='bottom' if eff >= 0 else 'top', fontsize=10)

for j in range(i + 1, len(axes)):
    axes[j].axis('off')

main_fig_path = FIG_DIR / f'main_stem_mean_ret_effect_{RUN_TIMESTAMP}.png'
save_figure(fig, main_fig_path)
plt.show()
print('saved:', main_fig_path)


## 3) 稳健性（Notebook 04：控制变量 / 子样本 / 置换 / 样本外）

重点关注：
- 控制变量回归（`weekday/month/year`）后，`stem=丙` 是否仍显著偏低
- 全局置换检验（是否存在“任意组效应”）
- 子样本与样本外（walk-forward）中符号是否稳定

新增（更保守 + 更可复现）：
- “控制变量残差”上的全局置换检验（先回归掉 `weekday/month/year`，再做 permutation）。
- HAC `maxlags` 敏感性 + block bootstrap（用于序列相关稳健性）。
- 跨指数复现（Meta 合并 + OOS 稳定性筛选）。


In [ ]:
# 04 控制变量回归：stem × ret_1d
# 说明：
# - p_value/q_value：组系数 vs 基准水平（不等价于“组 vs 全样本均值”）
# - p_value_effect/q_value_effect：组 vs 全样本边际均值（ret_1d 为 contrast；up 为 delta-method 近似）

controls_bing_rows = []
controls_sig_rows = []

for ts_code in TS_CODES:
    ctrl = read_controls(ts_code, group_col='stem', target='ret_1d')
    if ctrl.empty:
        continue

    bing = ctrl[ctrl['group_value'] == '丙']
    if len(bing) == 1:
        r = bing.iloc[0]
        controls_bing_rows.append(
            {
                'ts_code': ts_code,
                'effect_bing_minus_all': float(r['marginal_effect_vs_all']),
                'p_value_effect': float(r.get('p_value_effect', np.nan)),
                'q_value_effect': float(r.get('q_value_effect', np.nan)),
            }
        )

    if 'q_value_effect' in ctrl.columns and 'p_value_effect' in ctrl.columns:
        sig = ctrl[np.isfinite(ctrl['q_value_effect']) & (ctrl['q_value_effect'] <= ALPHA_Q)].copy()
        sig = sig.sort_values(['q_value_effect', 'p_value_effect'])
    else:
        sig = ctrl.iloc[0:0].copy()
    for _, r in sig.iterrows():
        controls_sig_rows.append(
            {
                'ts_code': ts_code,
                'group_value': r['group_value'],
                'effect': float(r['marginal_effect_vs_all']),
                'p_value_effect': float(r.get('p_value_effect', np.nan)),
                'q_value_effect': float(r.get('q_value_effect', np.nan)),
            }
        )

controls_bing_df = pd.DataFrame.from_records(controls_bing_rows)
if controls_bing_df.empty:
    controls_bing_df = pd.DataFrame(columns=['ts_code', 'effect_bing_minus_all', 'p_value_effect', 'q_value_effect'])
else:
    controls_bing_df = controls_bing_df.sort_values('ts_code')

controls_sig_df = pd.DataFrame.from_records(controls_sig_rows)
if controls_sig_df.empty:
    controls_sig_df = pd.DataFrame(columns=['ts_code', 'group_value', 'effect', 'p_value_effect', 'q_value_effect'])
else:
    controls_sig_df = controls_sig_df.sort_values(['ts_code', 'q_value_effect', 'p_value_effect'])

print('控制变量回归（04）: stem×ret_1d 中 q_effect<=', ALPHA_Q, '的项（若为空则表示未通过阈值）')
controls_sig_df if not controls_sig_df.empty else '(无)'


In [ ]:
# 图：04（控制变量）stem 的 ret_1d effect（组边际均值 - 全样本边际均值） + 误差条

n_plots = len(TS_CODES)
ncols = 2
nrows = int(np.ceil(n_plots / ncols))
fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(14, 4.8 * nrows), constrained_layout=True)
axes = np.atleast_1d(axes).ravel()

for i, ts_code in enumerate(TS_CODES):
    ax = axes[i]
    ctrl = read_controls(ts_code, group_col='stem', target='ret_1d')
    if ctrl.empty:
        ax.set_title(f'{ts_code}（缺少 controls）')
        ax.axis('off')
        continue

    ctrl['group_value'] = pd.Categorical(ctrl['group_value'], categories=STEMS_ORDER, ordered=True)
    ctrl = ctrl.sort_values('group_value')

    x = ctrl['group_value'].astype(str).tolist()
    y = ctrl['marginal_effect_vs_all'].to_numpy(dtype=float)
    se = ctrl.get('se_effect', pd.Series([np.nan] * len(ctrl))).to_numpy(dtype=float)
    qv = ctrl.get('q_value_effect', pd.Series([np.nan] * len(ctrl))).to_numpy(dtype=float)

    colors = []
    for gv, q in zip(x, qv):
        if gv == '丙':
            colors.append('tab:red')
        elif np.isfinite(q) and q <= ALPHA_Q:
            colors.append('tab:orange')
        else:
            colors.append('tab:blue')

    ax.bar(x, y, color=colors, alpha=0.85)
    if np.isfinite(se).any():
        ax.errorbar(x, y, yerr=se, fmt='none', ecolor='black', elinewidth=1, capsize=2, alpha=0.8)

    ax.axhline(0.0, color='black', lw=1)
    ax.set_title(f'04 控制变量：stem 对 ret_1d 的 effect（{ts_code}）')
    ax.set_ylabel('marginal effect (ret_1d)')
    ax.set_xlabel('stem')

    for xi, yi, qi in zip(x, y, qv):
        if np.isfinite(qi) and qi <= ALPHA_Q:
            ax.text(xi, yi, f'q={qi:.3g}', ha='center', va='bottom' if yi >= 0 else 'top', fontsize=10)

for j in range(i + 1, len(axes)):
    axes[j].axis('off')

ctrl_fig_path = FIG_DIR / f'controls_stem_ret_1d_effect_{RUN_TIMESTAMP}.png'
save_figure(fig, ctrl_fig_path)
plt.show()
print('saved:', ctrl_fig_path)


In [ ]:
# 子样本：丙 的 effect（按年份段）

subs = read_subsample_bing()
if subs.empty:
    print('缺少 subsample_bing.csv（如未运行 04，可忽略）')
else:
    subs_view = subs.copy()
    subs_view['effect_bp'] = subs_view['effect_bing_minus_all'].map(lambda x: float(x) * 10000)
    fig, ax = plt.subplots(figsize=(12, 5))
    sns.barplot(
        data=subs_view,
        x='ts_code',
        y='effect_bp',
        hue='bucket',
        order=TS_CODES,
        hue_order=['2010-2015', '2016-2020', '2021-now'],
        ax=ax,
    )
    ax.axhline(0.0, color='black', lw=1)
    ax.set_title('子样本：stem=丙 的 mean_ret effect（bp/日）')
    ax.set_xlabel('ts_code')
    ax.set_ylabel('effect (bp per day)')
    ax.legend(title='bucket', loc='best')

    subs_fig_path = FIG_DIR / f'subsample_bing_effect_bp_{RUN_TIMESTAMP}.png'
    save_figure(fig, subs_fig_path)
    plt.show()
    print('saved:', subs_fig_path)


In [ ]:
# 置换检验（全局）：是否存在“任意组效应”

perm = read_perm_global()
if perm.empty:
    print('缺少 perm_global.csv（如未运行 04，可忽略）')
else:
    view = perm.copy()
    view = view.sort_values(['p_empirical', 'ts_code', 'group_col', 'target'])
    view

# 控制变量残差置换检验（更保守）：stem × ret_resid
perm_resid = read_perm_global_controls_resid()
if perm_resid.empty:
    print('缺少 perm_global_controls_resid.csv（如未运行 04 的残差置换，可忽略）')
else:
    perm_resid = perm_resid.sort_values(['p_empirical', 'ts_code'])
    perm_resid



In [ ]:
# 样本外（walk-forward，按年）：丙 的 OOS 差异符号是否稳定

wf_rows = []
wf_all = []

for ts_code in TS_CODES:
    wf = read_walk_forward(ts_code)
    if wf.empty:
        continue
    wf_all.append(wf.assign(ts_code=ts_code))

    oos = wf.dropna(subset=['oos_bing_mean_ret_diff']).copy()
    if oos.empty:
        continue

    n = int(len(oos))
    n_neg = int((oos['oos_bing_mean_ret_diff'] < 0).sum())
    p_sign = float(stats.binomtest(n_neg, n=n, p=0.5, alternative='two-sided').pvalue) if n > 0 else np.nan

    wf_rows.append(
        {
            'ts_code': ts_code,
            'n_years': n,
            'neg_years': n_neg,
            'neg_ratio': n_neg / n if n else np.nan,
            'mean_oos_diff': float(oos['oos_bing_mean_ret_diff'].mean()),
            'p_value_sign_test': p_sign,
        }
    )

wf_summary_df = pd.DataFrame.from_records(wf_rows)
if wf_summary_df.empty:
    wf_summary_df = pd.DataFrame(
        columns=['ts_code', 'n_years', 'neg_years', 'neg_ratio', 'mean_oos_diff', 'p_value_sign_test']
    )
else:
    wf_summary_df = wf_summary_df.sort_values('ts_code')
wf_summary_df if not wf_summary_df.empty else '(缺少 walk_forward_*.csv；如未运行 04，可忽略)'


In [ ]:
# 图：OOS 年度差异（丙）

if wf_all:
    wf_all_df = pd.concat(wf_all, ignore_index=True)
    fig, ax = plt.subplots(figsize=(12, 5))
    sns.lineplot(
        data=wf_all_df,
        x='oos_year',
        y='oos_bing_mean_ret_diff',
        hue='ts_code',
        marker='o',
        ax=ax,
    )
    ax.axhline(0.0, color='black', lw=1)
    ax.set_title('Walk-forward（按年）：stem=丙 的 OOS mean_ret 差异')
    ax.set_xlabel('oos_year')
    ax.set_ylabel('oos_bing_mean_ret_diff (ret_1d)')

    wf_fig_path = FIG_DIR / f'walk_forward_bing_oos_diff_{RUN_TIMESTAMP}.png'
    save_figure(fig, wf_fig_path)
    plt.show()
    print('saved:', wf_fig_path)
else:
    print('缺少 walk_forward_*.csv（如未运行 04，可忽略）')


### 04j 共振检验（`jiazi_idx`；k=5/6；`ret_1d`）

说明：读取 04 的落盘结果（`data/clean_us/robustness/resonance_*`），并生成诊断图：
- `jiazi_idx` 上的观测均值 vs 谐波重建（bp/日）
- `jiazi_idx` 的频谱（FFT；仅作诊断）
- 10×12 嵌入热力图：观测 vs 谐波重建（不可达格子为 NaN）


In [ ]:
RESONANCE_PERIOD = 60
RESONANCE_KS = [5, 6]


def read_resonance_harmonic(ts_code: str) -> pd.DataFrame:
    p1 = ROBUST_DIR / f'resonance_harmonic_k56_{ts_code}_ret_1d.csv'
    df = safe_read_csv(p1)
    if not df.empty:
        return df
    p2 = ROBUST_DIR / 'resonance_harmonic_k56_ret_1d.csv'
    df2 = safe_read_csv(p2)
    if not df2.empty and 'ts_code' in df2.columns:
        return df2[df2['ts_code'] == ts_code].copy()
    return pd.DataFrame()


def read_resonance_after_additive(ts_code: str) -> pd.DataFrame:
    p1 = ROBUST_DIR / f'resonance_after_additive_{ts_code}_ret_1d.csv'
    df = safe_read_csv(p1)
    if not df.empty:
        return df
    p2 = ROBUST_DIR / 'resonance_after_additive_ret_1d.csv'
    df2 = safe_read_csv(p2)
    if not df2.empty and 'ts_code' in df2.columns:
        return df2[df2['ts_code'] == ts_code].copy()
    return pd.DataFrame()


def read_resonance_meta() -> pd.DataFrame:
    return safe_read_csv(ROBUST_DIR / 'resonance_meta_k56_ret_1d.csv')


def harmonic_reconstruct(i: np.ndarray, row: pd.Series) -> np.ndarray:
    # harmonic-only component (centered later)
    out = np.zeros_like(i, dtype=float)
    for k in RESONANCE_KS:
        beta_s = float(row.get(f'beta_s{k}', np.nan))
        beta_c = float(row.get(f'beta_c{k}', np.nan))
        if not (np.isfinite(beta_s) and np.isfinite(beta_c)):
            continue
        ang = 2.0 * np.pi * k * i / float(RESONANCE_PERIOD)
        out = out + beta_s * np.sin(ang) + beta_c * np.cos(ang)
    return out


# Summary tables
resonance_rows = []
SHOW_TS_FOR_FIGS = 'spx' if 'spx' in TS_CODES else (TS_CODES[0] if TS_CODES else None)

for ts_code in TS_CODES:
    r = read_resonance_harmonic(ts_code)
    if r.empty:
        continue

    row = r.iloc[0]
    ra = read_resonance_after_additive(ts_code)
    row_add = ra.iloc[0] if (isinstance(ra, pd.DataFrame) and (not ra.empty)) else pd.Series(dtype=float)

    # Be defensive about types
    n_days_val = row.get('n_days', np.nan)
    try:
        n_days = int(n_days_val)
    except Exception:
        n_days = ''

    resonance_rows.append(
        {
            'ts_code': ts_code,
            'n_days': n_days,
            'n_min_jiazi': int(row.get('n_min_jiazi', np.nan)) if np.isfinite(float(row.get('n_min_jiazi', np.nan))) else '',
            'n_max_jiazi': int(row.get('n_max_jiazi', np.nan)) if np.isfinite(float(row.get('n_max_jiazi', np.nan))) else '',
            'n_missing_jiazi': int(row.get('n_missing_jiazi', np.nan)) if np.isfinite(float(row.get('n_missing_jiazi', np.nan))) else '',
            'p_wald_k56': float(row.get('p_wald_k56', np.nan)),
            'q_wald_k56': float(row.get('q_wald_k56', np.nan)),
            'p_wald_k56_resid_additive': float(row_add.get('p_wald_k56_resid_additive', np.nan)),
            'amp_5_bp': float(row.get('amp_5', np.nan)) * 10000.0,
            'amp_6_bp': float(row.get('amp_6', np.nan)) * 10000.0,
            'delta_r2': float(row.get('delta_r2', np.nan)),
        }
    )

resonance_summary_df = pd.DataFrame.from_records(resonance_rows)

# If q is missing (e.g., only per-ts_code files exist), compute a small-family BH-FDR here.
if not resonance_summary_df.empty:
    p = resonance_summary_df['p_wald_k56'].to_numpy(dtype=float)
    if 'q_wald_k56' not in resonance_summary_df.columns or np.nanmax(np.abs(resonance_summary_df['q_wald_k56'].to_numpy(dtype=float))) == 0 or np.all(~np.isfinite(resonance_summary_df['q_wald_k56'].to_numpy(dtype=float))):
        resonance_summary_df['q_wald_k56'] = fdr_bh(np.nan_to_num(p, nan=1.0))

    p2 = resonance_summary_df['p_wald_k56_resid_additive'].to_numpy(dtype=float)
    resonance_summary_df['q_wald_k56_resid_additive'] = fdr_bh(np.nan_to_num(p2, nan=1.0))

resonance_meta_df = read_resonance_meta()

from IPython.display import display
if not resonance_meta_df.empty:
    display(resonance_meta_df)
if not resonance_summary_df.empty:
    display(resonance_summary_df)
else:
    print('未检测到 resonance 输出（请先 Run All notebooks_US/04f_resonance_harmonics.ipynb）。')


# Figures (save; only show for SPX (if available))
resonance_fig_paths: dict[str, dict[str, Path]] = {}

for ts_code in TS_CODES:
    r = read_resonance_harmonic(ts_code)
    if r.empty:
        continue

    row = r.iloc[0]
    data_path = CLEAN_DIR / f'market_ganzhi_{ts_code}.csv.gz'
    df = safe_read_csv(data_path, dtype={'trade_date': str})
    if df.empty:
        continue

    df = df.dropna(subset=['ret_1d', 'jiazi_idx', 'stem', 'branch']).copy()
    df['jiazi_idx'] = pd.to_numeric(df['jiazi_idx'], errors='coerce')
    df = df.dropna(subset=['jiazi_idx']).copy()
    df['jiazi_idx'] = df['jiazi_idx'].astype(int)
    df = df[df['jiazi_idx'].between(0, RESONANCE_PERIOD - 1)].copy()

    # 1D by jiazi_idx
    g = (
        df.groupby('jiazi_idx')
        .agg(n=('ret_1d', 'size'), mean_ret=('ret_1d', 'mean'), std_ret=('ret_1d', 'std'))
        .reset_index()
        .sort_values('jiazi_idx')
    )
    g['se_ret'] = g['std_ret'] / np.sqrt(g['n'].clip(lower=1))

    x = np.arange(RESONANCE_PERIOD, dtype=float)
    fit = harmonic_reconstruct(x, row)

    # Center both (fit is harmonic-only; obs includes intercept/controls)
    obs = g.set_index('jiazi_idx').reindex(range(RESONANCE_PERIOD))['mean_ret'].to_numpy(dtype=float)
    obs_centered = obs - np.nanmean(obs)
    fit_centered = fit - float(np.mean(fit))

    # Fitcurve
    fig1, ax1 = plt.subplots(figsize=(12, 4))
    ax1.plot(x, fit_centered * 10000.0, color='#C44E52', lw=2, label='harmonic(k=5/6) recon (centered)')
    ax1.scatter(g['jiazi_idx'], (g['mean_ret'] - np.nanmean(obs)) * 10000.0, s=25, color='#4C72B0', label='observed mean_ret by jiazi_idx (centered)')
    ax1.axhline(0.0, color='black', lw=1)
    ax1.set_title(f'{ts_code} | Resonance fit curve (ret_1d; bp/day; centered)')
    ax1.set_xlabel('jiazi_idx (0..59)')
    ax1.set_ylabel('bp / day')
    ax1.legend(loc='best', fontsize=9)

    p_fit = FIG_DIR / f'resonance_fitcurve_{ts_code}_ret_1d_{RUN_TIMESTAMP}.png'
    save_figure(fig1, p_fit)
    if SHOW_TS_FOR_FIGS is not None and ts_code == SHOW_TS_FOR_FIGS:
        plt.show()
    plt.close(fig1)

    # Spectrum (diagnostic FFT on obs_centered)
    y = np.nan_to_num(obs_centered, nan=0.0)
    fft = np.fft.rfft(y)
    amp = (2.0 / len(y)) * np.abs(fft)
    ks = np.arange(len(amp))  # 0..30

    fig2, ax2 = plt.subplots(figsize=(12, 4))
    ax2.bar(ks[1:], amp[1:] * 10000.0, color='#55A868')
    for k in RESONANCE_KS:
        ax2.axvline(k, color='#C44E52', lw=2)
        ax2.text(k + 0.1, float(np.max(amp[1:]) * 10000.0) * 0.95, f'k={k}', color='#C44E52')
    ax2.set_title(f'{ts_code} | FFT spectrum of mean_ret by jiazi_idx (bp/day; diagnostic)')
    ax2.set_xlabel('k (cycles per 60)')
    ax2.set_ylabel('amplitude (bp / day)')

    p_spec = FIG_DIR / f'resonance_spectrum_{ts_code}_ret_1d_{RUN_TIMESTAMP}.png'
    save_figure(fig2, p_spec)
    if SHOW_TS_FOR_FIGS is not None and ts_code == SHOW_TS_FOR_FIGS:
        plt.show()
    plt.close(fig2)

    # Embed heatmaps: observed vs harmonic recon (bp/day; centered)
    obs_heat = (
        df.groupby(['stem', 'branch'])['ret_1d']
        .mean()
        .reset_index()
        .pivot(index='stem', columns='branch', values='ret_1d')
        .reindex(index=STEMS_ORDER, columns=BRANCHES_ORDER)
    )
    obs_heat_bp = (obs_heat - float(np.nanmean(obs_heat))) * 10000.0

    fit_map = pd.DataFrame(index=STEMS_ORDER, columns=BRANCHES_ORDER, data=np.nan)
    for i_idx in range(RESONANCE_PERIOD):
        stem = STEMS_ORDER[i_idx % 10]
        branch = BRANCHES_ORDER[i_idx % 12]
        fit_map.loc[stem, branch] = fit_centered[i_idx] * 10000.0

    vmax = float(np.nanmax(np.abs(np.concatenate([obs_heat_bp.to_numpy().ravel(), fit_map.to_numpy().ravel()]))))
    if not np.isfinite(vmax) or vmax <= 0:
        vmax = 1.0

    fig3, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
    sns.heatmap(obs_heat_bp, center=0.0, cmap='RdBu_r', vmin=-vmax, vmax=vmax, ax=axes[0])
    axes[0].set_title('Observed (mean_ret; centered; bp/day)')
    axes[0].set_xlabel('branch')
    axes[0].set_ylabel('stem')

    sns.heatmap(fit_map, center=0.0, cmap='RdBu_r', vmin=-vmax, vmax=vmax, ax=axes[1])
    axes[1].set_title('Harmonic recon (k=5/6; centered; bp/day)')
    axes[1].set_xlabel('branch')
    axes[1].set_ylabel('')

    fig3.suptitle(f'{ts_code} | 10×12 embedding (unreachable cells = NaN)')

    p_emb = FIG_DIR / f'resonance_embed_heatmap_{ts_code}_ret_1d_{RUN_TIMESTAMP}.png'
    save_figure(fig3, p_emb)
    if SHOW_TS_FOR_FIGS is not None and ts_code == SHOW_TS_FOR_FIGS:
        plt.show()
    plt.close(fig3)

    resonance_fig_paths[ts_code] = {'fitcurve': p_fit, 'spectrum': p_spec, 'embed': p_emb}

    print('saved:', p_fit)
    print('saved:', p_spec)
    print('saved:', p_emb)


## 4) 一页结论表（建议用于对外沟通/快速复盘）

把关键证据链放在同一张表：
- 03（无控制）里 `stem=丙` 的样本内效应与 `q`
- 04（控制变量）里 `stem=丙` 的效应与 `q_effect`
- 全局置换检验（`stem×ret_1d`）的经验 p 值
- walk-forward 的符号稳定性（负年份占比 + 简单符号检验 p 值）
- 控制变量残差置换检验（`stem×ret_resid`）的经验 p 值（更保守）


In [ ]:
perm = read_perm_global()
required_cols = {'group_col', 'target', 'ts_code', 'p_empirical'}
if (not perm.empty) and required_cols.issubset(set(perm.columns)):
    perm_stem_ret = perm[(perm['group_col'] == 'stem') & (perm['target'] == 'ret_1d')][['ts_code', 'p_empirical']].copy()
else:
    perm_stem_ret = pd.DataFrame(columns=['ts_code', 'p_empirical'])

one_page = coverage_df.merge(main_bing_df, on='ts_code', how='left', suffixes=('', '_main'))
one_page = one_page.merge(
    controls_bing_df.rename(
        columns={
            'effect_bing_minus_all': 'effect_bing_minus_all_controls',
            'p_value_effect': 'p_value_effect_controls',
            'q_value_effect': 'q_value_effect_controls',
        }
    ),
    on='ts_code',
    how='left',
)
one_page = one_page.merge(perm_stem_ret, on='ts_code', how='left')

perm_resid = read_perm_global_controls_resid()
if (not perm_resid.empty) and {'ts_code','p_empirical'}.issubset(set(perm_resid.columns)):
    perm_stem_ret_resid = perm_resid[(perm_resid['group_col'] == 'stem') & (perm_resid['target'] == 'ret_resid')][['ts_code', 'p_empirical']].copy()
    perm_stem_ret_resid = perm_stem_ret_resid.rename(columns={'p_empirical': 'p_empirical_controls_resid'})
else:
    perm_stem_ret_resid = pd.DataFrame(columns=['ts_code', 'p_empirical_controls_resid'])

one_page = one_page.merge(perm_stem_ret_resid, on='ts_code', how='left')
one_page = one_page.merge(
    wf_summary_df.rename(
        columns={
            'neg_ratio': 'wf_neg_ratio',
            'p_value_sign_test': 'wf_p_value_sign_test',
            'mean_oos_diff': 'wf_mean_oos_diff',
        }
    ),
    on='ts_code',
    how='left',
)

one_page = one_page[
    [
        'ts_code',
        'n_days',
        'start_date',
        'end_date',
        'effect_bing_minus_all',
        'q_value',
        'effect_bing_minus_all_controls',
        'q_value_effect_controls',
        'p_empirical',
        'p_empirical_controls_resid',
        'wf_neg_ratio',
        'wf_p_value_sign_test',
    ]
].copy()

from IPython.display import display
display(one_page)


# ==============================
# 跨指数复现（Meta + OOS 稳定性）
# ==============================

# 1) Meta（固定效应）：使用 controls_{ts_code}_stem_ret_1d.csv 的 effect + se
meta_rows = []
long_rows = []
for ts_code in TS_CODES:
    ctrl = read_controls(ts_code, group_col='stem', target='ret_1d')
    if ctrl.empty:
        continue

    if not {'group_value', 'marginal_effect_vs_all', 'se_effect'}.issubset(set(ctrl.columns)):
        continue

    sub = ctrl[['group_value', 'marginal_effect_vs_all', 'se_effect']].copy()
    sub = sub.rename(columns={'marginal_effect_vs_all': 'effect', 'se_effect': 'se'})
    sub['ts_code'] = ts_code
    long_rows.append(sub)

if long_rows:
    long_df = pd.concat(long_rows, ignore_index=True)
else:
    long_df = pd.DataFrame(columns=['ts_code', 'group_value', 'effect', 'se'])

meta_records = []
for stem in STEMS_ORDER:
    sub = long_df[long_df['group_value'].astype(str) == stem].dropna(subset=['effect', 'se']).copy()
    sub = sub[np.isfinite(sub['se'].to_numpy(dtype=float)) & (sub['se'].to_numpy(dtype=float) > 0)].copy()

    k = int(len(sub))
    if k == 0:
        meta_records.append(
            {
                'group_value': stem,
                'effect_meta': np.nan,
                'se_meta': np.nan,
                'z_meta': np.nan,
                'p_meta': np.nan,
                'I2': np.nan,
                'n_indices_used': 0,
                'sign_consistent_count': 0,
            }
        )
        continue

    eff = sub['effect'].to_numpy(dtype=float)
    se = sub['se'].to_numpy(dtype=float)
    w = 1.0 / (se**2)

    eff_meta = float(np.sum(w * eff) / np.sum(w))
    se_meta = float(np.sqrt(1.0 / np.sum(w)))
    z_meta = float(eff_meta / se_meta) if se_meta > 0 else np.nan
    p_meta = float(2.0 * stats.norm.sf(abs(z_meta))) if np.isfinite(z_meta) else np.nan

    # heterogeneity
    q = float(np.sum(w * (eff - eff_meta) ** 2))
    df_q = max(1, k - 1)
    I2 = float(max(0.0, (q - df_q) / q)) if q > 0 else 0.0

    sign_target = float(np.sign(eff_meta))
    sign_consistent = int(np.sum(np.sign(eff) == sign_target)) if sign_target != 0 else 0

    meta_records.append(
        {
            'group_value': stem,
            'effect_meta': eff_meta,
            'se_meta': se_meta,
            'z_meta': z_meta,
            'p_meta': p_meta,
            'I2': I2,
            'n_indices_used': k,
            'sign_consistent_count': sign_consistent,
        }
    )

meta_df = pd.DataFrame.from_records(meta_records)
if not meta_df.empty:
    p = meta_df['p_meta'].to_numpy(dtype=float)
    ok = np.isfinite(p)
    q = np.full_like(p, np.nan, dtype=float)
    if ok.sum() > 0:
        q[ok] = fdr_bh(p[ok])
    meta_df['q_meta'] = q

meta_path = ROBUST_DIR / 'meta_controls_stem_ret_1d.csv'
meta_df.to_csv(meta_path, index=False)
print('saved:', meta_path)

# 2) OOS 稳定性（按 stem × 指数）：基于 walk_forward_{ts_code}_stem_oos_effects.csv
meta_sign_map = {r.group_value: float(np.sign(r.effect_meta)) if np.isfinite(r.effect_meta) else np.nan for r in meta_df.itertuples(index=False)}

oos_records = []
for ts_code in TS_CODES:
    oos = read_walk_forward_oos_effects(ts_code)
    if oos.empty:
        continue

    oos = oos.dropna(subset=['oos_year', 'stem', 'oos_effect']).copy()

    for stem, sub in oos.groupby('stem'):
        sub = sub.dropna(subset=['oos_effect'])
        n = int(len(sub))
        if n == 0:
            continue

        neg_years = int((sub['oos_effect'] < 0).sum())
        pos_years = int((sub['oos_effect'] > 0).sum())
        neg_ratio = neg_years / n
        pos_ratio = pos_years / n

        sign_target = meta_sign_map.get(str(stem), np.nan)
        if not np.isfinite(sign_target) or sign_target == 0:
            match_years = np.nan
            match_ratio = np.nan
            p_sign = np.nan
        else:
            match_years = int((sub['oos_effect'] * sign_target > 0).sum())
            match_ratio = match_years / n
            p_sign = float(stats.binomtest(match_years, n=n, p=0.5, alternative='two-sided').pvalue)

        oos_records.append(
            {
                'ts_code': ts_code,
                'group_value': str(stem),
                'n_years': n,
                'neg_ratio': neg_ratio,
                'pos_ratio': pos_ratio,
                'match_ratio': match_ratio,
                'p_sign': p_sign,
            }
        )

oos_stability_df = pd.DataFrame.from_records(oos_records)
oos_path = ROBUST_DIR / 'oos_stability_stem_ret_1d.csv'
oos_stability_df.to_csv(oos_path, index=False)
print('saved:', oos_path)

# 3) “正式结论”筛选：q_meta + 方向一致 + OOS 稳定
pass_records = []
for r in meta_df.itertuples(index=False):
    stem = str(r.group_value)
    k = int(getattr(r, 'n_indices_used', 0))
    if k <= 0 or (not np.isfinite(getattr(r, 'q_meta', np.nan))):
        continue

    pass_q = bool(getattr(r, 'q_meta') <= ALPHA_Q)

    thr_dir = int(np.ceil(0.75 * k))
    pass_dir = (float(np.sign(getattr(r, 'effect_meta'))) != 0) and (int(getattr(r, 'sign_consistent_count')) >= thr_dir)

    sub = oos_stability_df[(oos_stability_df['group_value'] == stem) & np.isfinite(oos_stability_df['match_ratio'])].copy()
    # 单指数稳定条件：match_ratio>=0.8 且 p_sign<=0.1
    sub['pass_oos_single'] = (sub['match_ratio'] >= 0.8) & (sub['p_sign'] <= 0.1)
    oos_pass_count = int(sub['pass_oos_single'].sum())

    thr_oos = int(np.ceil(0.75 * k))
    pass_oos = oos_pass_count >= thr_oos

    pass_all = pass_q and pass_dir and pass_oos

    pass_records.append(
        {
            'group_value': stem,
            'effect_meta': float(getattr(r, 'effect_meta')),
            'q_meta': float(getattr(r, 'q_meta')),
            'n_indices_used': k,
            'sign_consistent_count': int(getattr(r, 'sign_consistent_count')),
            'oos_pass_count': oos_pass_count,
            'pass_q_meta': pass_q,
            'pass_direction': pass_dir,
            'pass_oos': pass_oos,
            'pass_all': pass_all,
        }
    )

pass_df = pd.DataFrame.from_records(pass_records)
meta_pass_df = pass_df[pass_df['pass_all']].copy() if not pass_df.empty else pd.DataFrame(columns=pass_df.columns)
meta_pass_path = ROBUST_DIR / 'meta_replication_pass.csv'
meta_pass_df.to_csv(meta_pass_path, index=False)
print('saved:', meta_pass_path)

print('跨指数复现通过项（若为空表示未通过阈值）：')
meta_pass_df if not meta_pass_df.empty else '(无)'


# ==============================
# 候选 stem 深挖（癸/丁）：解释“Meta 显著但 OOS 不稳”
# ==============================

CANDIDATE_STEMS = ['癸', '丁']
BREAK_YEAR = 2020
N_PERM_YEAR_BREAK = 20000
RNG = np.random.default_rng(20260214)


def _plot_oos_bar(df: pd.DataFrame, title: str, path: Path) -> None:
    if df is None or df.empty:
        return
    tmp = df.sort_values('oos_year').copy()
    years = tmp['oos_year'].astype(int).to_numpy()
    vals = tmp['oos_effect'].astype(float).to_numpy()
    colors = ['#d62728' if v < 0 else '#1f77b4' for v in vals]

    fig, ax = plt.subplots(figsize=(10, 3))
    ax.bar(years, vals, color=colors, alpha=0.85)
    ax.axhline(0.0, color='black', linewidth=1)
    ax.set_title(title)
    ax.set_xlabel('OOS 年')
    ax.set_ylabel('OOS 效应（组均值 - 当年全 stem 均值）')
    ax.set_xticks(years)
    ax.tick_params(axis='x', rotation=45)
    save_figure(fig, path)
    plt.close(fig)


def _plot_oos_line_two(df_raw: pd.DataFrame, df_ctrl: pd.DataFrame, title: str, path: Path) -> None:
    if df_raw is None or df_raw.empty or df_ctrl is None or df_ctrl.empty:
        return

    r = df_raw[['oos_year', 'oos_effect']].rename(columns={'oos_effect': 'raw'}).copy()
    c = df_ctrl[['oos_year', 'oos_effect']].rename(columns={'oos_effect': 'controls'}).copy()
    m = r.merge(c, on='oos_year', how='inner').sort_values('oos_year')
    if m.empty:
        return

    years = m['oos_year'].astype(int).to_numpy()
    fig, ax = plt.subplots(figsize=(10, 3))
    ax.plot(years, m['raw'].to_numpy(dtype=float), marker='o', label='raw')
    ax.plot(years, m['controls'].to_numpy(dtype=float), marker='o', label='controls-only residual (weekday/month)')
    ax.axhline(0.0, color='black', linewidth=1)
    ax.set_title(title)
    ax.set_xlabel('OOS 年')
    ax.set_ylabel('OOS 效应')
    ax.set_xticks(years)
    ax.tick_params(axis='x', rotation=45)
    ax.legend(loc='best')
    save_figure(fig, path)
    plt.close(fig)


def _year_break_test(df: pd.DataFrame, break_year: int, n_perm: int, rng: np.random.Generator) -> dict:
    if df is None or df.empty:
        return {}

    tmp = df.dropna(subset=['oos_year', 'oos_effect']).copy()
    tmp['oos_year'] = tmp['oos_year'].astype(int)
    tmp = tmp.sort_values('oos_year')

    pre = tmp[tmp['oos_year'] < int(break_year)]['oos_effect'].astype(float).to_numpy()
    post = tmp[tmp['oos_year'] >= int(break_year)]['oos_effect'].astype(float).to_numpy()
    if pre.size == 0 or post.size == 0:
        return {}

    diff_obs = float(post.mean() - pre.mean())
    vals = tmp['oos_effect'].astype(float).to_numpy()
    n_total = int(vals.size)
    n_pre = int(pre.size)

    count = 1
    for _ in range(int(n_perm)):
        perm = rng.permutation(n_total)
        pre_idx = perm[:n_pre]
        post_idx = perm[n_pre:]
        diff = float(vals[post_idx].mean() - vals[pre_idx].mean())
        if abs(diff) >= abs(diff_obs):
            count += 1

    p = count / (int(n_perm) + 1)

    return {
        'n_years': n_total,
        'n_pre': n_pre,
        'n_post': int(post.size),
        'mean_pre': float(pre.mean()),
        'mean_post': float(post.mean()),
        'diff_post_minus_pre': diff_obs,
        'p_perm_year_shuffle': float(p),
    }


pass_stems = set(meta_pass_df['group_value'].astype(str).tolist()) if isinstance(meta_pass_df, pd.DataFrame) and (not meta_pass_df.empty) else set()
candidate_stems = [s for s in CANDIDATE_STEMS if (s in STEMS_ORDER) and (s not in pass_stems)]
print('candidate_stems =', candidate_stems)

candidate_figs_raw: dict[str, list[Path]] = {}
candidate_figs_ctrl: dict[str, list[Path]] = {}
candidate_figs_overlay: dict[str, list[Path]] = {}
candidate_figs_avg: dict[str, Path] = {}


# 1) raw OOS yearly: table + per-index plots
raw_rows = []
for ts_code in TS_CODES:
    oos = read_walk_forward_oos_effects(ts_code)
    if oos.empty or (not candidate_stems):
        continue

    oos = oos.dropna(subset=['oos_year', 'stem', 'oos_effect']).copy()
    oos['stem'] = oos['stem'].astype(str)
    oos = oos[oos['stem'].isin(candidate_stems)].copy()
    if oos.empty:
        continue

    oos['meta_sign'] = oos['stem'].map(lambda s: float(meta_sign_map.get(str(s), np.nan)))
    oos['sign_match'] = np.where(
        np.isfinite(oos['meta_sign']) & (oos['meta_sign'] != 0),
        (oos['oos_effect'] * oos['meta_sign'] > 0),
        np.nan,
    )

    raw_rows.append(oos)

    for stem in candidate_stems:
        sub = oos[oos['stem'] == stem][['oos_year', 'oos_effect']].copy()
        if sub.empty:
            continue
        fig_path = FIG_DIR / f'candidate_raw_oos_{stem}_{ts_code}_{RUN_TIMESTAMP}.png'
        _plot_oos_bar(sub, title=f'raw OOS 年度效应：stem={stem} | {ts_code}', path=fig_path)
        candidate_figs_raw.setdefault(stem, []).append(fig_path)

raw_candidate_df = (
    pd.concat(raw_rows, ignore_index=True)
    if raw_rows
    else pd.DataFrame(columns=['ts_code', 'oos_year', 'stem', 'oos_effect', 'meta_sign', 'sign_match'])
)
raw_candidate_path = ROBUST_DIR / 'candidate_oos_yearly_raw_stem_ret_1d.csv'
raw_candidate_df.to_csv(raw_candidate_path, index=False)
print('saved:', raw_candidate_path)

# Cross-index average (visual aid; not an independent test)
for stem in candidate_stems:
    sub = raw_candidate_df[raw_candidate_df['stem'] == stem].copy()
    if sub.empty:
        continue
    avg = sub.groupby('oos_year', as_index=False).agg(oos_effect=('oos_effect', 'mean'))
    fig_path = FIG_DIR / f'candidate_raw_oos_{stem}_avg_{RUN_TIMESTAMP}.png'
    _plot_oos_bar(avg, title=f'raw OOS 跨指数平均：stem={stem}', path=fig_path)
    candidate_figs_avg[stem] = fig_path


# 2) raw：flip years summary (relative to Meta sign)
flip_records = []
for stem in candidate_stems:
    sign_target = float(meta_sign_map.get(str(stem), np.nan))
    for ts_code in TS_CODES:
        sub = raw_candidate_df[(raw_candidate_df['stem'] == stem) & (raw_candidate_df['ts_code'] == ts_code)].copy()
        if sub.empty:
            continue

        sub = sub.sort_values('oos_year')
        years = sub['oos_year'].astype(int).tolist()
        vals = sub['oos_effect'].astype(float).to_numpy()
        n = int(len(vals))
        neg_years = int((vals < 0).sum())
        pos_years = int((vals > 0).sum())

        if np.isfinite(sign_target) and sign_target != 0:
            match_years = int((vals * sign_target > 0).sum())
            match_ratio = match_years / n
            p_sign = float(stats.binomtest(match_years, n=n, p=0.5, alternative='two-sided').pvalue)
            flip_years = [y for y, v in zip(years, vals) if v * sign_target < 0]
        else:
            match_ratio = np.nan
            p_sign = np.nan
            flip_years = []

        seq = ''.join('+' if v > 0 else '-' if v < 0 else '0' for v in vals.tolist())

        flip_records.append(
            {
                'stem': stem,
                'ts_code': ts_code,
                'meta_sign': sign_target,
                'n_years': n,
                'pos_years': pos_years,
                'neg_years': neg_years,
                'match_ratio_raw': match_ratio,
                'p_sign_raw': p_sign,
                'flip_years_raw': ','.join([str(y) for y in flip_years]) if flip_years else '',
                'sign_seq_raw': seq,
            }
        )

flip_df = pd.DataFrame.from_records(flip_records)
flip_path = ROBUST_DIR / 'candidate_oos_flip_years_stem_ret_1d.csv'
flip_df.to_csv(flip_path, index=False)
print('saved:', flip_path)


# 3) controls-only residual OOS (if available): table + plots + stability
ctrl_rows = []
for ts_code in TS_CODES:
    oos = read_walk_forward_oos_effects_controls(ts_code)
    if oos.empty or (not candidate_stems):
        continue

    oos = oos.dropna(subset=['oos_year', 'stem', 'oos_effect']).copy()
    oos['stem'] = oos['stem'].astype(str)
    oos = oos[oos['stem'].isin(candidate_stems)].copy()
    if oos.empty:
        continue

    ctrl_rows.append(oos)

    for stem in candidate_stems:
        sub_ctrl = oos[oos['stem'] == stem][['oos_year', 'oos_effect']].copy()
        sub_raw = raw_candidate_df[(raw_candidate_df['stem'] == stem) & (raw_candidate_df['ts_code'] == ts_code)][
            ['oos_year', 'oos_effect']
        ].copy()

        if not sub_ctrl.empty:
            fig_path = FIG_DIR / f'candidate_controls_oos_{stem}_{ts_code}_{RUN_TIMESTAMP}.png'
            _plot_oos_bar(sub_ctrl, title=f'controls-only residual OOS：stem={stem} | {ts_code}', path=fig_path)
            candidate_figs_ctrl.setdefault(stem, []).append(fig_path)

        if (not sub_ctrl.empty) and (not sub_raw.empty):
            fig_path = FIG_DIR / f'candidate_controls_vs_raw_{stem}_{ts_code}_{RUN_TIMESTAMP}.png'
            _plot_oos_line_two(sub_raw, sub_ctrl, title=f'raw vs controls OOS：stem={stem} | {ts_code}', path=fig_path)
            candidate_figs_overlay.setdefault(stem, []).append(fig_path)

controls_candidate_df = (
    pd.concat(ctrl_rows, ignore_index=True)
    if ctrl_rows
    else pd.DataFrame(columns=['ts_code', 'oos_year', 'stem', 'oos_effect'])
)
controls_candidate_path = ROBUST_DIR / 'candidate_oos_yearly_controls_stem_ret_1d.csv'
controls_candidate_df.to_csv(controls_candidate_path, index=False)
print('saved:', controls_candidate_path)


# OOS stability on controls residuals (all stems; per index)
oos_controls_records = []
for ts_code in TS_CODES:
    oos = read_walk_forward_oos_effects_controls(ts_code)
    if oos.empty:
        continue

    oos = oos.dropna(subset=['oos_year', 'stem', 'oos_effect']).copy()
    for stem, sub in oos.groupby('stem'):
        sub = sub.dropna(subset=['oos_effect'])
        n = int(len(sub))
        if n == 0:
            continue

        neg_years = int((sub['oos_effect'] < 0).sum())
        pos_years = int((sub['oos_effect'] > 0).sum())
        neg_ratio = neg_years / n
        pos_ratio = pos_years / n

        sign_target = meta_sign_map.get(str(stem), np.nan)
        if not np.isfinite(sign_target) or float(sign_target) == 0:
            match_ratio = np.nan
            p_sign = np.nan
        else:
            match_years = int((sub['oos_effect'] * float(sign_target) > 0).sum())
            match_ratio = match_years / n
            p_sign = float(stats.binomtest(match_years, n=n, p=0.5, alternative='two-sided').pvalue)

        oos_controls_records.append(
            {
                'ts_code': ts_code,
                'group_value': str(stem),
                'n_years': n,
                'neg_ratio': neg_ratio,
                'pos_ratio': pos_ratio,
                'match_ratio': match_ratio,
                'p_sign': p_sign,
            }
        )

oos_controls_df = pd.DataFrame.from_records(oos_controls_records)
oos_controls_path = ROBUST_DIR / 'oos_stability_stem_ret_1d_controls.csv'
oos_controls_df.to_csv(oos_controls_path, index=False)
print('saved:', oos_controls_path)


# 4) Break test (pre/post 2020; year-level permutation; diagnostic only)
break_records = []
for stem in candidate_stems:
    for ts_code in TS_CODES:
        sub_raw = raw_candidate_df[(raw_candidate_df['stem'] == stem) & (raw_candidate_df['ts_code'] == ts_code)][
            ['oos_year', 'oos_effect']
        ].copy()
        if not sub_raw.empty:
            st = _year_break_test(sub_raw, break_year=BREAK_YEAR, n_perm=N_PERM_YEAR_BREAK, rng=RNG)
            if st:
                break_records.append({'stem': stem, 'ts_code': ts_code, 'metric': 'raw', **st})

        sub_ctrl = controls_candidate_df[(controls_candidate_df['stem'] == stem) & (controls_candidate_df['ts_code'] == ts_code)][
            ['oos_year', 'oos_effect']
        ].copy()
        if not sub_ctrl.empty:
            st = _year_break_test(sub_ctrl, break_year=BREAK_YEAR, n_perm=N_PERM_YEAR_BREAK, rng=RNG)
            if st:
                break_records.append({'stem': stem, 'ts_code': ts_code, 'metric': 'controls_resid', **st})

break_df = pd.DataFrame.from_records(break_records)
break_path = ROBUST_DIR / 'candidate_break_test_stem_ret_1d.csv'
break_df.to_csv(break_path, index=False)
print('saved:', break_path)


# 5) Window sensitivity (controls residual; relies on 04 outputs)
sensitivity_rows = []
for ts_code in TS_CODES:
    sweep = read_walk_forward_oos_effects_controls_sweep(ts_code)
    if sweep.empty:
        continue

    sweep = sweep.dropna(subset=['oos_year', 'stem', 'oos_effect', 'train_years']).copy()
    sweep['stem'] = sweep['stem'].astype(str)
    sweep['train_years'] = sweep['train_years'].astype(int)

    for (ty, stem), sub in sweep.groupby(['train_years', 'stem']):
        sub = sub.dropna(subset=['oos_effect'])
        n = int(len(sub))
        if n == 0:
            continue

        sign_target = float(meta_sign_map.get(str(stem), np.nan))
        if np.isfinite(sign_target) and sign_target != 0:
            match_years = int((sub['oos_effect'].astype(float) * sign_target > 0).sum())
            match_ratio = match_years / n
            p_sign = float(stats.binomtest(match_years, n=n, p=0.5, alternative='two-sided').pvalue)
        else:
            match_ratio = np.nan
            p_sign = np.nan

        sensitivity_rows.append(
            {
                'ts_code': ts_code,
                'train_years': int(ty),
                'group_value': str(stem),
                'n_years': n,
                'match_ratio': match_ratio,
                'p_sign': p_sign,
            }
        )

sensitivity_df = pd.DataFrame.from_records(sensitivity_rows)
for ts_code in TS_CODES:
    out = sensitivity_df[sensitivity_df['ts_code'] == ts_code].copy()
    out_path = ROBUST_DIR / f'oos_window_sensitivity_controls_{ts_code}_stem_ret_1d.csv'
    out.to_csv(out_path, index=False)
    print('saved:', out_path)

summary_df = pd.DataFrame()
if not sensitivity_df.empty:
    keep_stems = ['丙', '癸', '丁']
    tmp = sensitivity_df[sensitivity_df['group_value'].isin(keep_stems)].copy()
    tmp['pass_single'] = (tmp['match_ratio'] >= 0.8) & (tmp['p_sign'] <= 0.1)
    summary_df = (
        tmp.groupby(['train_years', 'group_value'])
        .agg(
            n_indices=('ts_code', 'nunique'),
            median_match_ratio=('match_ratio', 'median'),
            pass_count=('pass_single', 'sum'),
        )
        .reset_index()
        .sort_values(['train_years', 'group_value'])
    )

summary_path = ROBUST_DIR / 'oos_window_sensitivity_summary_controls_stem_ret_1d.csv'
summary_df.to_csv(summary_path, index=False)
print('saved:', summary_path)


## 5) 导出 Markdown 报告（写入 `data/clean_us/report/`）

Run All 后会自动生成：
- `report_*.md`
- 对应的图表文件（在 `figures/`）


In [ ]:
def series_to_brief_kv(s: pd.Series) -> str:
    parts = []
    for k, v in s.items():
        parts.append(f'- {k}: {v}')
    return '\n'.join(parts)


key_takeaways = []

sig_main = main_sig_df.copy() if isinstance(main_sig_df, pd.DataFrame) else pd.DataFrame()
sig_controls = controls_sig_df.copy() if isinstance(controls_sig_df, pd.DataFrame) else pd.DataFrame()

meta_pass_df = globals().get('meta_pass_df', pd.DataFrame())

if isinstance(meta_pass_df, pd.DataFrame) and (not meta_pass_df.empty):
    vals = meta_pass_df['group_value'].astype(str).tolist()
    key_takeaways.append(f"跨指数复现（Meta+OOS）通过：{','.join(vals)}")
elif not sig_controls.empty:
    key_takeaways.append(
        f"控制变量回归（04，stem×ret_1d）在 q_effect<= {ALPHA_Q} 下出现信号：" +
        '，'.join([f"{r.ts_code}:{r.group_value}" for r in sig_controls.itertuples(index=False)])
    )
elif not sig_main.empty:
    key_takeaways.append(
        f"样本内（03，stem×mean_ret）在 q<= {ALPHA_Q} 下出现信号：" +
        '，'.join([f"{r.ts_code}:{r.group_value}" for r in sig_main.itertuples(index=False)])
    )
else:
    key_takeaways.append(f"在当前阈值（q<= {ALPHA_Q}）下，未观察到稳定通过的 stem×mean_ret 信号。")

report_lines = []
report_lines.append(f"# 流日干支对美国宽基指数涨跌的影响：一键结论报告")
report_lines.append('')
report_lines.append(f"- 生成时间：`{RUN_TIMESTAMP}`")
report_lines.append(f"- 样本指数：`{', '.join(TS_CODES)}`")
report_lines.append(f"- 显著性阈值：`q <= {ALPHA_Q}`（BH-FDR）")
report_lines.append('')

report_lines.append('## 关键结论（简版）')
for t in key_takeaways:
    report_lines.append(f"- {t}")
report_lines.append('')

report_lines.append('## 数据覆盖')
report_lines.append(df_to_markdown_table(coverage_df))
report_lines.append('')

if 'stem_dist' in globals() and 'branch_dist' in globals():
    report_lines.append('### 干支日历分布（交易日，sanity check）')
    report_lines.append(df_to_markdown_table(stem_dist))
    report_lines.append('')
    report_lines.append(df_to_markdown_table(branch_dist))
    report_lines.append('')

report_lines.append('## 一页结论表（stem=丙）')
report_lines.append(df_to_markdown_table(one_page))
report_lines.append('')

report_lines.append('## 主分析（03：无控制变量）')
report_lines.append(f"- 图：`{main_fig_path.name}`")
report_lines.append(f"![](figures/{main_fig_path.name})")
report_lines.append('')
report_lines.append('显著项（q<=阈值；若为空表示未通过阈值）：')
report_lines.append(df_to_markdown_table(main_sig_df) if isinstance(main_sig_df, pd.DataFrame) else '(无)')
report_lines.append('')

report_lines.append('全量扫描（每个 group_type×metric 的最小 q；不代表通过阈值）：')
report_lines.append(df_to_markdown_table(top_scan_df, max_rows=500) if isinstance(top_scan_df, pd.DataFrame) else '(无)')
report_lines.append('')

report_lines.append('通过阈值的项（q<=阈值）：')
if isinstance(top_scan_pass_df, pd.DataFrame) and (not top_scan_pass_df.empty):
    report_lines.append(df_to_markdown_table(top_scan_pass_df, max_rows=500))
else:
    report_lines.append('(无)')
report_lines.append('')

report_lines.append('## 稳健性（04）')
report_lines.append('### 04a 控制变量回归（weekday/month/year）')
report_lines.append(f"- 图：`{ctrl_fig_path.name}`")
report_lines.append(f"![](figures/{ctrl_fig_path.name})")
report_lines.append('')
report_lines.append('显著项（q_effect<=阈值；若为空表示未通过阈值）：')
report_lines.append(df_to_markdown_table(controls_sig_df) if isinstance(controls_sig_df, pd.DataFrame) else '(无)')
report_lines.append('')

if 'subs_fig_path' in globals():
    report_lines.append('### 04b 子样本（年份段）')
    report_lines.append(f"- 图：`{subs_fig_path.name}`")
    report_lines.append(f"![](figures/{subs_fig_path.name})")
    report_lines.append('')

if not perm.empty:
    report_lines.append('### 04c 置换检验（全局）')
    report_lines.append('> p_empirical 越小表示“存在任意组效应”的证据越强。')
    report_lines.append(df_to_markdown_table(perm.sort_values(['p_empirical']).head(20), max_rows=20))
    report_lines.append('')

# 04c2 控制变量残差置换检验（更保守）
perm_resid = globals().get('perm_resid', pd.DataFrame())
if isinstance(perm_resid, pd.DataFrame) and (not perm_resid.empty):
    report_lines.append('### 04c2 置换检验（控制变量残差）')
    report_lines.append('> 先回归掉 `weekday/month/year`，对 `stem × ret_resid` 做全局置换检验（更保守）。')
    report_lines.append(df_to_markdown_table(perm_resid.sort_values(['p_empirical']).head(20), max_rows=20))
    report_lines.append('')

if 'wf_fig_path' in globals():
    report_lines.append('### 04d 样本外（walk-forward，按年）')
    report_lines.append(f"- 图：`{wf_fig_path.name}`")
    report_lines.append(f"![](figures/{wf_fig_path.name})")
    report_lines.append('')
    report_lines.append('汇总：')
    report_lines.append(df_to_markdown_table(wf_summary_df))
    report_lines.append('')


# 04e HAC maxlags 敏感性（仅展示 stem=丙）
hac_rows = []
for ts_code in TS_CODES:
    sens = read_hac_sensitivity(ts_code)
    if sens.empty:
        continue
    view = sens[sens['group_value'].astype(str) == '丙'].copy()
    if view.empty:
        continue
    view = view.sort_values('maxlags')
    for _, r in view.iterrows():
        hac_rows.append(
            {
                'ts_code': ts_code,
                'maxlags': int(r['maxlags']),
                'effect': float(r['marginal_effect_vs_all']),
                'p_value_effect': float(r.get('p_value_effect', np.nan)),
                'q_value_effect': float(r.get('q_value_effect', np.nan)),
            }
        )

hac_view = pd.DataFrame.from_records(hac_rows)
if not hac_view.empty:
    report_lines.append('### 04e HAC `maxlags` 敏感性（stem=丙）')
    report_lines.append(df_to_markdown_table(hac_view.sort_values(['ts_code', 'maxlags'])))
    report_lines.append('')

# 04f block bootstrap（controls 残差；仅展示 stem=丙）
boot_rows = []
for ts_code in TS_CODES:
    boot = read_block_bootstrap(ts_code)
    if boot.empty:
        continue
    view = boot[boot['group_value'].astype(str) == '丙']
    if view.empty:
        continue
    r = view.iloc[0]
    boot_rows.append(
        {
            'ts_code': ts_code,
            'effect_obs': float(r.get('effect_obs', np.nan)),
            'boot_ci_low': float(r.get('boot_ci_low', np.nan)),
            'boot_ci_high': float(r.get('boot_ci_high', np.nan)),
            'p_boot_two_sided': float(r.get('p_boot_two_sided', np.nan)),
        }
    )

boot_view = pd.DataFrame.from_records(boot_rows)
if not boot_view.empty:
    report_lines.append('### 04f block bootstrap（controls 残差；stem=丙）')
    report_lines.append(df_to_markdown_table(boot_view.sort_values('ts_code')))
    report_lines.append('')

# 04g 跨指数复现（Meta + OOS）
meta_df = globals().get('meta_df', pd.DataFrame())
meta_pass_df = globals().get('meta_pass_df', pd.DataFrame())
oos_stability_df = globals().get('oos_stability_df', pd.DataFrame())

if isinstance(meta_df, pd.DataFrame) and (not meta_df.empty):
    report_lines.append('### 04g 跨指数复现（Meta + OOS）')
    if isinstance(meta_pass_df, pd.DataFrame) and (not meta_pass_df.empty):
        report_lines.append('通过“q_meta + 方向一致 + OOS 稳定”的项：')
        report_lines.append(df_to_markdown_table(meta_pass_df.sort_values('q_meta')))
    else:
        report_lines.append('在当前阈值下：未观察到通过“q_meta + 方向一致 + OOS 稳定”的项。')
    report_lines.append('')

    report_lines.append('Meta（10 天干全量）：')
    meta_view = meta_df.copy()
    if 'q_meta' in meta_view.columns:
        meta_view = meta_view.sort_values(['q_meta', 'p_meta'])
    report_lines.append(df_to_markdown_table(meta_view, max_rows=20))
    report_lines.append('')

    if isinstance(meta_pass_df, pd.DataFrame) and (not meta_pass_df.empty) and isinstance(oos_stability_df, pd.DataFrame) and (not oos_stability_df.empty):
        pass_stems = meta_pass_df['group_value'].astype(str).tolist()
        oos_view = oos_stability_df[oos_stability_df['group_value'].isin(pass_stems)].copy()
        report_lines.append('OOS 稳定性（仅展示通过项）：')
        report_lines.append(df_to_markdown_table(oos_view.sort_values(['group_value', 'ts_code']), max_rows=200))
        report_lines.append('')

# 04h 候选 stem 深挖（癸/丁）：诊断“Meta 显著但 OOS 不稳”
candidate_stems = globals().get('candidate_stems', [])
flip_df = globals().get('flip_df', pd.DataFrame())
break_df = globals().get('break_df', pd.DataFrame())
oos_controls_df = globals().get('oos_controls_df', pd.DataFrame())
summary_df = globals().get('summary_df', pd.DataFrame())
candidate_figs_raw = globals().get('candidate_figs_raw', {})
candidate_figs_overlay = globals().get('candidate_figs_overlay', {})
candidate_figs_avg = globals().get('candidate_figs_avg', {})

if isinstance(candidate_stems, list) and candidate_stems:
    report_lines.append('### 04h 候选 stem 深挖（癸/丁：Meta 显著但 OOS 不稳的诊断）')
    report_lines.append('> 本节仅做诊断，不改变“正式结论”筛选规则。')
    cand_str = '，'.join([str(s) for s in candidate_stems])
    report_lines.append(f'候选 stem：{cand_str}')
    report_lines.append('')

    if isinstance(flip_df, pd.DataFrame) and (not flip_df.empty):
        report_lines.append('raw OOS：翻符号年份（相对 Meta 方向）：')
        report_lines.append(df_to_markdown_table(flip_df.sort_values(['stem', 'ts_code']), max_rows=200))
        report_lines.append('')

    if isinstance(oos_controls_df, pd.DataFrame) and (not oos_controls_df.empty):
        view = oos_controls_df[oos_controls_df['group_value'].isin(candidate_stems)].copy()
        if not view.empty:
            report_lines.append('controls-only residual OOS：稳定性指标（相对 Meta 方向）：')
            report_lines.append(df_to_markdown_table(view.sort_values(['group_value', 'ts_code']), max_rows=200))
            report_lines.append('')
    else:
        report_lines.append('未检测到 controls-only residual OOS 输出（请先 Run All notebooks_US/04d_oos_walk_forward.ipynb）。')
        report_lines.append('')

    if isinstance(break_df, pd.DataFrame) and (not break_df.empty):
        break_year = int(globals().get('BREAK_YEAR', 2020))
        report_lines.append(f'断点对比（按年，{break_year} 前后；年份层面置换，仅作提示）：')
        report_lines.append(df_to_markdown_table(break_df.sort_values(['stem', 'ts_code', 'metric']), max_rows=200))
        report_lines.append('')

    if isinstance(summary_df, pd.DataFrame) and (not summary_df.empty):
        report_lines.append('训练窗敏感性（controls-only residual；相对 Meta 方向）：')
        report_lines.append(df_to_markdown_table(summary_df, max_rows=200))
        report_lines.append('')

    for stem in candidate_stems:
        avg_p = candidate_figs_avg.get(stem) if isinstance(candidate_figs_avg, dict) else None
        if avg_p is not None:
            name = Path(avg_p).name
            report_lines.append(f'图：raw OOS 跨指数平均（stem={stem}）：`{name}`')
            report_lines.append(f'![](figures/{name})')
            report_lines.append('')

        figs = []
        label = ''
        if isinstance(candidate_figs_overlay, dict) and candidate_figs_overlay.get(stem):
            figs = candidate_figs_overlay.get(stem, [])
            label = 'raw vs controls OOS（每指数）'
        elif isinstance(candidate_figs_raw, dict) and candidate_figs_raw.get(stem):
            figs = candidate_figs_raw.get(stem, [])
            label = 'raw OOS（每指数）'

        if figs:
            report_lines.append(f'图：{label}（stem={stem}）')
            for p in figs:
                name = Path(p).name
                report_lines.append(f'- `{name}`')
                report_lines.append(f'![](figures/{name})')
            report_lines.append('')

# 04i Phase 2：year_element 交互（全局 gate → 才看局部 cell）
phase2_gate_df = read_phase2_gate_summary()
if phase2_gate_df.empty:
    report_lines.append('### 04i Phase 2：立春年 year_element 交互（day_group × year_element；全局 gate）')
    report_lines.append('未检测到 Phase 2 输出（缺少 `interaction_gate_summary_ret_1d.csv`；请先 Run All notebooks_US/04e_phase2_interaction_gate.ipynb）。')
    report_lines.append('')
else:
    report_lines.append('### 04i Phase 2：立春年 year_element 交互（day_group × year_element；全局 gate）')
    report_lines.append('> 说明：采用层级检验（全局 gate → 才看局部 cell），防止在高维 cell 中事后挑显著。')

    view = phase2_gate_df.copy()
    if 'pass_gate' in view.columns:
        view['pass_gate'] = view['pass_gate'].astype(str).str.lower().isin(['true', '1', 'yes'])
    sort_cols = [c for c in ['pass_gate', 'q_meta_interaction', 'p_meta_interaction'] if c in view.columns]
    if sort_cols:
        view = view.sort_values(sort_cols, ascending=[False] + [True] * (len(sort_cols) - 1))

    report_lines.append('Gate 汇总（跨指数 Fisher 合并 + BH-FDR；仅 3 个 day_group）：')
    report_lines.append(df_to_markdown_table(view, max_rows=20))
    report_lines.append('')

    passed = view[view['pass_gate'] == True].copy() if 'pass_gate' in view.columns else view.iloc[0:0].copy()
    if passed.empty:
        report_lines.append('在当前阈值下：未通过 gate，因此不输出局部 heatmap/候选。')
        report_lines.append('')
    else:
        for day_group in passed['day_group'].astype(str).tolist():
            parts = []
            for ts_code in TS_CODES:
                df_cell = read_phase2_cell_effects(ts_code, day_group)
                if df_cell.empty:
                    continue
                df_cell = df_cell.copy()
                df_cell['ts_code'] = ts_code
                parts.append(df_cell)

            if not parts:
                report_lines.append(f'- `{day_group}`：gate 通过，但缺少 cell-level 输出（请检查 notebooks_US/04e_phase2_interaction_gate.ipynb 的运行）。')
                report_lines.append('')
                continue

            cell = pd.concat(parts, ignore_index=True)
            if 'marginal_effect_vs_all' not in cell.columns:
                report_lines.append(f'- `{day_group}`：缺少 marginal_effect_vs_all，无法生成 heatmap/候选。')
                report_lines.append('')
                continue

            cell['marginal_effect_vs_all'] = pd.to_numeric(cell['marginal_effect_vs_all'], errors='coerce')
            if 'n_cell' in cell.columns:
                cell['n_cell'] = pd.to_numeric(cell['n_cell'], errors='coerce')

            pivot = cell.pivot_table(
                index=['group_value', 'year_element'],
                columns='ts_code',
                values='marginal_effect_vs_all',
                aggfunc='first',
            )

            mean_eff = pivot.mean(axis=1, skipna=True)
            k_used = pivot.notna().sum(axis=1)
            mean_sign = np.sign(mean_eff)

            base = np.sign(pivot).eq(mean_sign, axis=0)
            non_zero = mean_sign.notna() & mean_sign.ne(0)
            sign_ok = base & non_zero.to_numpy()[:, None]
            sign_consistent = sign_ok.sum(axis=1)

            out = pd.DataFrame(
                {
                    'group_value': [i[0] for i in mean_eff.index],
                    'year_element': [i[1] for i in mean_eff.index],
                    'k_used': k_used.to_numpy(dtype=int),
                    'mean_effect': mean_eff.to_numpy(dtype=float),
                    'mean_effect_bp': mean_eff.to_numpy(dtype=float) * 10000,
                    'sign_consistent_count': sign_consistent.to_numpy(dtype=int),
                }
            )

            if 'n_cell' in cell.columns:
                nstat = (
                    cell.groupby(['group_value', 'year_element'], dropna=False)['n_cell']
                    .agg(n_median='median', n_min='min')
                    .reset_index()
                )
                out = out.merge(nstat, on=['group_value', 'year_element'], how='left')

            out['min_sign_required'] = np.ceil(0.75 * out['k_used']).astype(int)
            out['pass_direction'] = (out['k_used'] > 0) & (out['sign_consistent_count'] >= out['min_sign_required'])

            cand = out[out['pass_direction']].copy()
            cand['abs_mean_bp'] = cand['mean_effect_bp'].abs()
            cand = cand.sort_values(['abs_mean_bp'], ascending=False)

            cand_path = ROBUST_DIR / f'interaction_candidates_{day_group}_ret_1d.csv'
            cand.to_csv(cand_path, index=False)
            print('saved:', cand_path)

            heat = out.pivot(index='group_value', columns='year_element', values='mean_effect_bp')
            heat = heat.reindex(columns=YEAR_ELEMENT_ORDER)
            if day_group == 'stem':
                heat = heat.reindex(index=STEMS_ORDER)
            elif day_group == 'branch':
                heat = heat.reindex(index=BRANCHES_ORDER)
            elif day_group == 'ganzhi_day' and 'jiazi_idx' in cell.columns:
                jidx = (
                    cell[['group_value', 'jiazi_idx']]
                    .dropna()
                    .drop_duplicates(subset=['group_value'])
                    .assign(jiazi_idx=lambda d: pd.to_numeric(d['jiazi_idx'], errors='coerce'))
                    .dropna()
                    .sort_values('jiazi_idx')
                )
                if not jidx.empty:
                    heat = heat.reindex(index=jidx['group_value'].astype(str).tolist())

            fig_h = max(4.0, 0.35 * max(1, heat.shape[0]))
            fig_w = 8.0 if day_group != 'ganzhi_day' else 9.5
            fig, ax = plt.subplots(figsize=(fig_w, fig_h))
            sns.heatmap(heat, center=0.0, cmap='RdBu_r', ax=ax)
            ax.set_title(f'Phase 2 heatmap（{day_group} × year_element；mean effect，bp/日；仅 gate 通过展示）')
            ax.set_xlabel('year_element')
            ax.set_ylabel(day_group)
            fig_path = FIG_DIR / f'phase2_interaction_heatmap_{day_group}_{RUN_TIMESTAMP}.png'
            save_figure(fig, fig_path)
            plt.show()
            print('saved:', fig_path)

            report_lines.append(f'局部 heatmap（{day_group}×year_element；bp/日；仅 gate 通过展示）：`{fig_path.name}`')
            report_lines.append(f'![](figures/{fig_path.name})')
            report_lines.append('')
            report_lines.append(f'候选（Exploratory；方向一致≥75%；按跨指数平均效应绝对值排序；已落盘 `{cand_path.name}`）：')
            report_lines.append(df_to_markdown_table(cand, max_rows=20))
            report_lines.append('')


# 04j Resonance: jiazi_idx harmonic k=5/6 (ret_1d)
resonance_summary_df = globals().get('resonance_summary_df', pd.DataFrame())
resonance_meta_df = globals().get('resonance_meta_df', pd.DataFrame())
resonance_fig_paths = globals().get('resonance_fig_paths', {})

report_lines.append('### 04j 60 甲子共振（jiazi_idx；k=5/6；ret_1d）')
report_lines.append('> 把“10×12 热力图条纹”回到 60 周期（jiazi_idx=0..59）上；主检验为 k=5/6 谐波项联合显著（HAC；含 controls=weekday/month/year）。')
report_lines.append('> 注意：即使显著，也只代表存在周期成分；不等价于机制/因果证明。')
report_lines.append('> 表中 `p_wald_k56_resid_additive` / `q_wald_k56_resid_additive` 为诊断：先回归掉 stem+branch 主效应后的残差再检验（Exploratory）。')

if isinstance(resonance_meta_df, pd.DataFrame) and (not resonance_meta_df.empty):
    report_lines.append('跨指数合并（Fisher）：')
    report_lines.append(df_to_markdown_table(resonance_meta_df, max_rows=5))
    report_lines.append('')

if isinstance(resonance_summary_df, pd.DataFrame) and (not resonance_summary_df.empty):
    report_lines.append('单指数结果：')
    report_lines.append(df_to_markdown_table(resonance_summary_df.sort_values('p_wald_k56'), max_rows=50))
    report_lines.append('')

    # Embed figures (prefer SPX if available)
    show_ts = None
    if isinstance(resonance_fig_paths, dict) and resonance_fig_paths:
        show_ts = 'spx' if 'spx' in resonance_fig_paths else sorted(resonance_fig_paths.keys())[0]

    if show_ts is not None:
        figs = resonance_fig_paths.get(show_ts, {})
        report_lines.append(f'图（示例：{show_ts}）：')
        for key in ['fitcurve', 'spectrum', 'embed']:
            p = figs.get(key)
            if isinstance(p, Path):
                report_lines.append(f"- `{p.name}`")
                report_lines.append(f"![](figures/{p.name})")
        report_lines.append('')

        # List other ts_codes (file names only)
        others = [k for k in resonance_fig_paths.keys() if k != show_ts]
        if others:
            report_lines.append('其余指数图表（已保存到 `data/clean_us/report/figures/`，此处不逐一嵌入）：')
            for k in others:
                figd = resonance_fig_paths.get(k, {})
                names = [v.name for v in figd.values() if isinstance(v, Path)]
                if names:
                    report_lines.append(f"- `{k}`: {', '.join(names)}")
            report_lines.append('')
else:
    report_lines.append('未检测到 resonance 输出（请先 Run All notebooks_US/04f_resonance_harmonics.ipynb）。')
    report_lines.append('')

report_lines.append('## 风险提示与下一步')
report_lines.append('- 多重比较与数据窥探：务必坚持 q 值阈值，并优先看跨指数一致性与样本外。')
report_lines.append('- 序列相关：已补充 HAC `maxlags` 敏感性 + block bootstrap（见 04e/04f），仍建议根据机器资源增加次数做复核。')
report_lines.append('- 更保守的稳健性：已在 controls 残差上做全局置换检验（见 04c2）。')
report_lines.append('')

report_md = '\n'.join(report_lines)
REPORT_PATH.write_text(report_md, encoding='utf-8')
print('saved report:', REPORT_PATH)

# notebook 内直接展示一份（便于 Run All 后阅读）
from IPython.display import Markdown, display

display(Markdown(report_md[:20000] + ('\n\n> 注：报告较长，此处仅展示前 20000 字。' if len(report_md) > 20000 else '')))
